# Chapter 8 — Optimization: The DevOps Agent, From Prototype to Production

Companion notebook to `skills/optimization/`. This is the chapter's capstone: the
DevOps agent that has grown across Chapters 2-7 (knowledge graph, memory,
reasoning, tool orchestration, self-evolution) now gets the engineering it needs
to run 24/7 at production scale.

**The three forces production collides with (Ch8):**

1. **Cost** — every workflow node on a frontier model is unsustainable at volume.
2. **Governance** — one unscoped graph query can traverse from a public catalog
   to employee records.
3. **Performance** — a graph traversal that is milliseconds in dev takes seconds
   at scale; a chain of specialist calls becomes a latency bottleneck.

**Running scenario:** a deployment event fires — `checkout-service` is updating
`stripe-python` from `3.2.1` to `3.3.0` in a fictional AWS account
(`123456789012`). We trace that single event through the optimized pipeline and
exercise all five Chapter-8 skills. Every AWS call is `moto`-mocked, so the
notebook runs with zero credentials; swap the `@mock_aws` decorator for your real
account to deploy.

## 1. Load the five optimization skills

Each skill ships a `lib.py`. Because they share the module name `lib`, we load
each from its own path under a distinct alias rather than a bare `import lib`.

In [ ]:
import sys, subprocess, json, os, importlib.util
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OPT = PROJECT_ROOT / 'skills' / 'optimization'

def _load(slug, alias):
    spec = importlib.util.spec_from_file_location(alias, OPT / slug / 'lib.py')
    mod = importlib.util.module_from_spec(spec)
    sys.modules[alias] = mod  # register so @dataclass can resolve __module__
    spec.loader.exec_module(mod)
    return mod

routing  = _load('model-routing-selector', 'routing_lib')
costperf = _load('cost-performance-scorer', 'costperf_lib')
govern   = _load('subgraph-access-control', 'govern_lib')
schema   = _load('schema-evolution-migrator', 'schema_lib')
budget   = _load('kv-cache-latency-budgeter', 'budget_lib')

print('Loaded 5 optimization skills from', OPT)

## 2. Selective Intelligence — route each node to the model it actually needs

**Why.** The horizontal workflow graph decomposes the agent into specialized
nodes. Most teams run every node on one frontier model; it works and it is wildly
expensive. Selective intelligence matches model capability to task complexity: a
fine-tuned 3B SLM classifies alerts (10-30x cheaper per token), a mid-tier 8B
model traverses the dependency graph, and only open-ended synthesis and causal
reasoning reach the frontier. The `model-routing-selector` skill re-derives the
book's `DEVOPS_MODEL_CONFIG` (Example 8-13) from each node's quality bar, so you
see *why* each node gets its model.

In [ ]:
nodes = ['AlertClassifier', 'QueryAnalyst', 'LogParser',
         'DependencyAnalyzer', 'CausalAttributionNode', 'PredictionSynthesis']

print(f"{'node':<24} {'strategy':<9} model")
print('-' * 66)
for n in nodes:
    r = routing.cheapest_meeting_bar(n)
    print(f"{n:<24} {r['strategy']:<9} {r['model']}")

pc = routing.pipeline_cost(0.30)
print(f"\nBlended pipeline cost reduction vs frontier-everywhere: {pc['reduction_pct']}%")
print(f"(book reports ~80% under token-weighting; equal-weight nodes here land near 70%)")

# The same decision through the CLI, proving harness portability
out = subprocess.run(
    [sys.executable, str(OPT / 'model-routing-selector' / 'cli.py'), 'route', 'CausalAttributionNode'],
    capture_output=True, text=True)
print('\nCLI route CausalAttributionNode ->', json.loads(out.stdout)['model'])

In [ ]:
# Verification gate: the router reproduces the book's assignment and the
# blended cost falls substantially.
assert routing.cheapest_meeting_bar('AlertClassifier')['model'] == 'llama-3.1-3b'
assert routing.cheapest_meeting_bar('CausalAttributionNode')['strategy'] == 'cascade'
assert routing.cheapest_meeting_bar('PredictionSynthesis')['model'] == 'claude-sonnet'
assert pc['reduction_pct'] >= 65.0
print('PASS: AlertClassifier on a 3B SLM, CausalAttributionNode cascaded, '
      f"synthesis on the frontier, blended cost down {pc['reduction_pct']}%.")

## 3. Measuring Cost-Performance — cost per successful completion

**Why.** Routing is only as good as the data behind it. The metric that matters
is not cost per token but **cost per successful completion**: a cheap model that
fails often can cost more per completed task than a reliable one. We wrap each
node's invocation in a `CostTracker` (Example 8-3), mirror the cost/latency
signals into `moto`-mocked CloudWatch (the shape real Bedrock/CloudWatch metrics
take), read the total back to prove the AWS seam, and score the policy.

In [ ]:
import datetime
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
os.environ.setdefault('AWS_ACCESS_KEY_ID', 'testing')
os.environ.setdefault('AWS_SECRET_ACCESS_KEY', 'testing')
import boto3
from moto import mock_aws

sample = json.loads((OPT / 'cost-performance-scorer' / 'sample-invocations.json').read_text())

@mock_aws
def measure_policy():
    cw = boto3.client('cloudwatch')
    now = datetime.datetime.now(datetime.timezone.utc)
    tracker = costperf.CostTracker()
    for row in sample['invocations']:
        inv = costperf.NodeInvocation(**row)
        tracker.log(inv)
        # emit the per-call cost/latency as CloudWatch metrics (real boto3 shape)
        cw.put_metric_data(Namespace='DevOpsAgent/Pipeline', MetricData=[
            {'MetricName': 'CostUSD',   'Value': inv.cost_usd,   'Timestamp': now},
            {'MetricName': 'LatencyMs', 'Value': inv.latency_ms, 'Timestamp': now},
        ])
    stats = cw.get_metric_statistics(
        Namespace='DevOpsAgent/Pipeline', MetricName='CostUSD',
        StartTime=now - datetime.timedelta(minutes=10),
        EndTime=now + datetime.timedelta(minutes=10),
        Period=3600, Statistics=['Sum', 'SampleCount'])
    cw_total = sum(dp['Sum'] for dp in stats['Datapoints'])
    return tracker, cw_total

tracker, cw_total = measure_policy()
report = costperf.score_policy(tracker)
print(f"{'node':<24} {'$/success':>11} {'succ%':>6} {'p95 ms':>8}")
print('-' * 52)
for r in report['nodes']:
    print(f"{r['node']:<24} {r['cost_per_success']:>11.6f} "
          f"{r['success_rate']*100:>5.0f}% {r['p95_latency_ms']:>8.1f}")
print(f"\nPipeline cost per success: ${report['pipeline_cost_per_success']}")
print(f"CloudWatch (moto) total cost read back: ${round(cw_total, 6)}")

inv_cmp = costperf.compare_cost_per_success(0.007, 0.60, 0.010, 0.98)
print(f"Cost-per-success inversion: cheaper per call = {inv_cmp['cheaper_per_call']}, "
      f"cheaper per success = {inv_cmp['cheaper_per_success']}")

In [ ]:
# Verification gate: cost-per-success computed, SLM cheaper than frontier per
# success, the CloudWatch seam agrees with the tracker, and the inversion holds.
ac = tracker.node_report('AlertClassifier')
ps = tracker.node_report('PredictionSynthesis')
assert ac['cost_per_success'] > 0
assert ac['cost_per_success'] < ps['cost_per_success']
assert abs(cw_total - report['pipeline_total_cost_usd']) < 1e-5
assert inv_cmp['cheaper_per_success'] == 'reliable'
print('PASS: cost-per-success measured; the SLM node is far cheaper per success '
      'than the frontier node; the moto CloudWatch total matches the tracker; '
      'the cheaper-per-call model loses on cost-per-success.')

## 4. Hardware Acceleration — dependency analysis within the latency budget

**Why.** Incident response is the agent's most latency-sensitive path. Blast-radius
analysis (PageRank over the dependency graph) is bound by memory bandwidth, a
natural fit for GPUs: on CPU it takes 3-5 seconds, on one GPU under 100 ms
(Example 8-14). And at the inference layer, **peak KV per active user — not model
size — is the binding concurrency constraint**; quantizing weights does not move
that ceiling, but bounding the cache (MEMENTO) does. The `kv-cache-latency-budgeter`
skill checks both.

In [ ]:
br = budget.blast_radius_report()
est = budget.estimate_gpu_time(35, 'betweenness_centrality')
print(f"Blast radius (Example 8-14): {br['cpu_seconds']} s CPU -> {br['gpu_result_ms']} ms GPU")
print(f"cuGraph estimate: 35 s CPU @ {est['speedup']}x -> {est['gpu_ms']} ms GPU")

bp = budget.budget_pipeline(n_slm_calls=5, n_graph_ops=1, parallel_factor=2.0)
print(f"\nPipeline latency (5 SLM calls + 1 graph op, 2x parallel): "
      f"{bp['estimated_ms']} ms; within 2s target: {bp['within_2s_target']}")

kv = budget.weight_quant_vs_kv_compression(gpu_memory_gb=80, model_weights_gb=16,
                                           peak_kv_per_user_gb=8,
                                           weight_quant_factor=2, kv_compression_factor=2)
print(f"\nConcurrency on an 80 GB H100:")
print(f"  baseline            : {kv['baseline_concurrency']} users")
print(f"  + weight quant (2x) : {kv['after_weight_quant']} users  (gain {kv['weight_quant_gain']})")
print(f"  + KV compress  (2x) : {kv['after_kv_compression']} users  (gain {kv['kv_compression_gain']})")
print(f"  winner: {kv['verdict']}  ->  bound the cache, not the weights")

In [ ]:
# Verification gate: blast radius is sub-100ms on GPU, the pipeline fits the 2s
# budget, and KV compression beats weight quantization for concurrency.
assert est['gpu_ms'] < 100
assert bp['within_2s_target']
assert kv['verdict'] == 'kv_compression'
assert kv['kv_compression_gain'] > kv['weight_quant_gain']
print('PASS: blast radius sub-100ms on GPU; the pipeline lands within the sub-2s '
      'budget; KV compression multiplies concurrency where weight quantization '
      'barely moves it.')

## 5. Governance for Infrastructure Data — scope what each persona sees

**Why.** Cost and speed make the agent viable; governance makes it safe to deploy.
The DevOps graph holds service configs, cost data, and on-call schedules with
employee names. The SRE persona needs full infrastructure visibility but no cost
data; the finance persona needs cost data but no deployment topology. The
`subgraph-access-control` skill generates the Neo4j policy, enforces security
transparency (out-of-scope nodes are invisible, not access-denied), and attaches
governance metadata to the execution graph as a compliance artifact.

In [ ]:
print('SRE role policy (Neo4j GRANT/DENY):')
print(govern.generate_policy('sre_oncall'))

audit = govern.audit_access('sre_oncall', ['Service', 'Library', 'Metric'])
print(f"\nSRE dependency query functionally complete: {audit['functionally_complete']}")
print(f"SRE can read cost_per_hour:      {govern.can_read('sre_oncall', 'Service', 'cost_per_hour')}")
print(f"finance can read cost_per_hour:  {govern.can_read('finance_analyst', 'Service', 'cost_per_hour')}")
print(f"finance can traverse Library:    {govern.can_traverse('finance_analyst', 'Library')}")

rec = govern.governance_metadata(
    data_sources_accessed=['infrastructure_graph', 'metrics_api'],
    access_role='sre_on_call', model_id='llama-3.1-8b-instruct',
    model_version='v2.3.1', decision_confidence=0.87, pii_accessed=False)
print('\nExecution-graph governance record (Example 8-6):')
print(json.dumps(rec, indent=2))

records = [rec, govern.governance_metadata(
    ['identity_store'], 'billing_agent', 'claude-sonnet', 'v1.0', 0.90, pii_accessed=True)]
pii_hits = govern.audit_query(records, pii_accessed=True)
print(f"\nAuditor query 'which decisions accessed PII last quarter': {len(pii_hits)} decision(s)")

In [ ]:
# Verification gate: least privilege enforced both ways, SRE reasoning not
# starved, the decision is attributed to a model, and the PII audit works.
assert not govern.can_read('sre_oncall', 'Service', 'cost_per_hour')
assert govern.can_read('finance_analyst', 'Service', 'cost_per_hour')
assert not govern.can_traverse('finance_analyst', 'Library')
assert not govern.can_traverse('sre_oncall', 'Employee')
assert audit['functionally_complete']
assert rec['model_id'] == 'llama-3.1-8b-instruct'
assert len(pii_hits) == 1
print('PASS: SRE cannot see cost data; finance cannot see deployment topology; '
      'Employee is invisible to both; SRE reasoning is not starved; every '
      'decision is attributed to its model; the PII audit is a graph query.')

## 6. Production Maintenance — schema evolution and the incremental update

**Why.** A graph in production is not static. The Chapter-7 improvement adds a
`MONITORED_BY` relationship; deploying it means coordinating three versioned
components — schema, data, and agent code — with N-1 compatibility so old code
keeps working. And when the `stripe-python` deployment event fires, the graph must
reflect it in seconds (LightRAG incremental merge), invalidating the old
dependency edge rather than deleting it so causal reasoning keeps its history. The
`schema-evolution-migrator` skill covers all of it.

In [ ]:
mig = schema.monitored_by_migration()
print(f"Migration {mig.version}: {mig.description}  (N-1 compatible: {mig.n1_compatible})")

manifest = schema.staged_rollout_manifest('2024-03-18-causal-attribution-v2')
v = schema.validate_manifest(manifest)
print(f"Deployment manifest phases: {[p['name'] for p in manifest['phases']]}")
print(f"Manifest valid: {v['valid']}")

event = json.loads((OPT / 'schema-evolution-migrator' / 'sample-deployment-event.json').read_text())
stmts = schema.incremental_merge_cypher(event)
print(f"\nIncremental merge for {event['service']} "
      f"(deps {', '.join(d['name'] for d in event['dependencies'])}): {len(stmts)} statements")

inval = schema.temporal_invalidate_cypher('checkout-service', 'stripe-python', '3.2.1', '3.3.0')
sets_invalid = 't_invalid = datetime()' in inval and 'DELETE' not in inval.upper()
print(f"Temporal invalidation stripe-python 3.2.1 -> 3.3.0 sets t_invalid, no delete: {sets_invalid}")

growth = schema.snapshot_growth_per_day(200, 5)
print(f"Snapshot growth (Fischer): 200 resources @ 5-min = {growth} nodes/day")

In [ ]:
# Verification gate: valid N-1 rollout, incremental upsert semantics, and the
# Fischer growth figure.
assert mig.n1_compatible
assert v['valid']
joined = '\n'.join(stmts)
assert 'ON CREATE SET' in joined and 'ON MATCH SET' in joined
assert 'NOT l.name IN $current_libs' in joined
assert growth == 57600
print('PASS: N-1 compatible migration + a valid four-phase staged rollout; the '
      'incremental merge upserts and invalidates-not-deletes; Fischer growth is '
      '57,600 nodes/day for one small cluster.')

## 7. The Completed Agent — one deployment event, all five skills

**Why.** Trace the single `checkout-service` / `stripe-python 3.2.1 -> 3.3.0`
event through the optimized pipeline (Example 8-16). Selective intelligence routes
each node; governance gates the infrastructure data; the schema skill records the
event incrementally; the budgeter confirms the latency; and CloudWatch (mocked)
carries the cost signal. This is the production-hardened agent.

In [ ]:
@mock_aws
def handle_deployment_event(event):
    cw = boto3.client('cloudwatch')
    now = datetime.datetime.now(datetime.timezone.utc)
    pipeline = ['AlertClassifier', 'LogParser', 'DependencyAnalyzer',
                'CausalAttributionNode', 'PredictionSynthesis']

    # 1. Selective intelligence — route each node (Pillar 8)
    routes = {n: routing.cheapest_meeting_bar(n) for n in pipeline}

    # 2. Governance — the agent acts under the SRE role; cost data is out of scope
    assert not govern.can_read('sre_oncall', 'Service', 'cost_per_hour')

    # 3. Memory — incremental KG update from the deployment event (Pillar 2)
    merge_stmts = schema.incremental_merge_cypher(event)

    # 4. Cost signals per node -> tracker + CloudWatch (illustrative, ~$0.002 total)
    node_signals = {
        'AlertClassifier':       (0.00007, 41.0),
        'LogParser':             (0.00021, 88.0),
        'DependencyAnalyzer':    (0.00030, 180.0),
        'CausalAttributionNode': (0.00050, 220.0),
        'PredictionSynthesis':   (0.00100, 710.0),
    }
    tracker = costperf.CostTracker()
    for n, (c, l) in node_signals.items():
        tracker.log(costperf.NodeInvocation(n, routes[n]['model'], 0, 0, c, l, True))
        cw.put_metric_data(Namespace='DevOpsAgent/Event', MetricData=[
            {'MetricName': 'CostUSD', 'Value': c, 'Timestamp': now},
            {'MetricName': 'LatencyMs', 'Value': l, 'Timestamp': now},
        ])

    # 5. Latency budget check (Pillar 8)
    lat = budget.budget_pipeline(n_slm_calls=5, n_graph_ops=1, parallel_factor=2.0)

    # 6. Governance record for the execution graph (Pillar 7 + Pillar 8)
    rec = govern.governance_metadata(
        ['infrastructure_graph', 'metrics_api'], 'sre_on_call',
        'llama-3.1-8b-instruct', 'v2.3.1', 0.87, pii_accessed=False)

    stats = cw.get_metric_statistics(
        Namespace='DevOpsAgent/Event', MetricName='CostUSD',
        StartTime=now - datetime.timedelta(minutes=10),
        EndTime=now + datetime.timedelta(minutes=10),
        Period=3600, Statistics=['Sum'])
    event_cost = sum(dp['Sum'] for dp in stats['Datapoints'])
    return {'routes': routes, 'merge_statements': len(merge_stmts),
            'latency': lat, 'governance': rec, 'event_cost_usd': event_cost}

result = handle_deployment_event(event)
print('Pipeline routing:')
for n in ['AlertClassifier', 'LogParser', 'DependencyAnalyzer',
          'CausalAttributionNode', 'PredictionSynthesis']:
    print(f"  {n:<24} -> {result['routes'][n]['model']}")
print(f"\nIncremental KG merge statements: {result['merge_statements']}")
print(f"Event cost (CloudWatch moto):    ${round(result['event_cost_usd'], 5)} "
      f"(book target ~$0.002 vs ~$0.01 frontier-everywhere)")
print(f"End-to-end latency estimate:     {result['latency']['estimated_ms']} ms "
      f"(within 2s: {result['latency']['within_2s_target']})")
print(f"Governance role:                 {result['governance']['access_role']}, "
      f"pii_accessed={result['governance']['pii_accessed']}")

In [ ]:
# Closing verdict: the production-hardened agent is routed, governed, within
# latency budget, and affordable.
pc = routing.pipeline_cost(0.30)
routed = pc['reduction_pct'] >= 65.0
governed = (not govern.can_read('sre_oncall', 'Service', 'cost_per_hour')
            and result['governance']['pii_accessed'] is False)
within_budget = result['latency']['within_2s_target']
affordable = result['event_cost_usd'] < 0.01

assert routed and governed and within_budget and affordable

print('=' * 68)
print('The production-hardened DevOps agent:')
print(f"  ROUTED     - blended cost {pc['reduction_pct']}% below frontier-everywhere")
print(f"  GOVERNED   - SRE role cannot read cost data; every decision is logged")
print(f"  IN BUDGET  - {result['latency']['estimated_ms']} ms end-to-end, within the sub-2s target")
print(f"  AFFORDABLE - ${round(result['event_cost_usd'], 5)} per event")
print('=' * 68)
print('Intelligent, self-improving, cost-effective, secure, and fast enough to '
      'operate in real time. The stripe-python 3.2.1 -> 3.3.0 upgrade that would '
      'have woken an SRE at 3 a.m. is now handled autonomously.')